In [6]:
"""
TransientFlowCylinder_TF2.py
============================
TensorFlow 2.x GPU-optimised PINN for transient laminar flow past a cylinder.

• Runs on GPU (tf.function + XLA jit_compile) when TensorFlow ≥ 2.x is present.
• Falls back to a numerically-identical NumPy/SciPy implementation when TF is
  absent, so the script is always executable and always produces all figures.
• pyDOE / pyDOE2 are replaced by scipy.stats.qmc.LatinHypercube, removing
  the external dependency entirely.

Figures produced per run
------------------------
  loss_convergence.png              – semi-log Adam loss curves
  inlet_profile_3d.png              – 3-D surface of inlet velocity profile  (NEW)
  collocation_distribution.png      – x-y scatter of all training points     (NEW)
  snapshot_t{T}_uvp.png             – contour/heatmap u,v,p fields           (NEW)
  snapshot_t{T}_streamlines.png     – streamline overlay on speed magnitude  (NEW)
  pressure_polar_t{T}.png           – p vs polar angle around cylinder       (NEW)
  pressure_history_probes.png       – time-series p at P1,P2,P3             (NEW+orig)
  uvp_t{NNN}.png                    – original scatter-field frames (51 total)
"""

# ── stdlib / third-party ──────────────────────────────────────────────────────
import os, sys, shutil, time, pickle, random
import numpy as np
import scipy.optimize
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from mpl_toolkits.mplot3d import Axes3D          # noqa: F401  (registers 3-D projection)
from scipy.stats.qmc import LatinHypercube

# ── TensorFlow (optional) ─────────────────────────────────────────────────────
try:
    import tensorflow as tf
    _TF_AVAILABLE = (int(tf.__version__.split('.')[0]) >= 2)
except ImportError:
    _TF_AVAILABLE = False

# ── Global seeds ──────────────────────────────────────────────────────────────
GLOBAL_SEED = 1234
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
if _TF_AVAILABLE:
    tf.random.set_seed(GLOBAL_SEED)
    for g in tf.config.list_physical_devices('GPU'):
        tf.config.experimental.set_memory_growth(g, True)
    DTYPE = tf.float32
else:
    DTYPE = np.float32

# ─────────────────────────────────────────────────────────────────────────────
# §1  Latin-Hypercube helper (replaces pyDOE)
# ─────────────────────────────────────────────────────────────────────────────
def lhs(d: int, n: int, seed: int = GLOBAL_SEED) -> np.ndarray:
    """Return an (n, d) LHS sample in [0, 1]^d."""
    return LatinHypercube(d=d, seed=seed).random(n=n).astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# §2  Geometry helpers  (identical to original)
# ─────────────────────────────────────────────────────────────────────────────
def DelSrcPT(XY_c, xc=0.0, yc=0.0, r=0.1):
    dst = ((XY_c[:, 0] - xc)**2 + (XY_c[:, 1] - yc)**2)**0.5
    return XY_c[dst > r]


def CartGrid(xmin, xmax, ymin, ymax, tmin, tmax, num_x, num_y, num_t):
    x = np.linspace(xmin, xmax, num_x)
    y = np.linspace(ymin, ymax, num_y)
    t = np.linspace(tmin, tmax, num_t)
    xxx, yyy, ttt = np.meshgrid(x, y, t)
    return (xxx.flatten()[:, None].astype(np.float32),
            yyy.flatten()[:, None].astype(np.float32),
            ttt.flatten()[:, None].astype(np.float32))


def GenCirclePT(xc, yc, r, tmin, tmax, num_r, num_t):
    theta = np.linspace(0.0, 2 * np.pi, num_r)
    x = r * np.cos(theta) + xc
    y = r * np.sin(theta) + yc
    t = np.linspace(tmin, tmax, num_t)
    xx, tt = np.meshgrid(x, t)
    yy, _  = np.meshgrid(y, t)
    return (xx.flatten()[:, None].astype(np.float32),
            yy.flatten()[:, None].astype(np.float32),
            tt.flatten()[:, None].astype(np.float32))


# ─────────────────────────────────────────────────────────────────────────────
# §3  Data generation  (identical to original, parameterised for ablations)
# ─────────────────────────────────────────────────────────────────────────────
def generate_data(n_base=80000, n_refine=15000, n_lw=3000, n_up=3000):
    xmax, tmax = 1.1, 0.5
    lb = np.array([0., 0., 0.], dtype=np.float32)
    ub = np.array([xmax, 0.41, tmax], dtype=np.float32)

    # Initial condition  (t=0 plane)
    x_IC, y_IC, t_IC = CartGrid(0, xmax, 0, 0.41, 0, 0, 81, 41, 1)
    IC = np.concatenate([x_IC, y_IC, t_IC], 1)
    IC = DelSrcPT(IC, xc=0.2, yc=0.2, r=0.05)

    # Upper / lower walls
    x_upb, y_upb, t_upb = CartGrid(0, xmax, 0.41, 0.41, 0, tmax, 81, 1, 41)
    x_lwb, y_lwb, t_lwb = CartGrid(0, xmax, 0,    0,    0, tmax, 81, 1, 41)
    wall_up = np.concatenate([x_upb, y_upb, t_upb], 1)
    wall_lw = np.concatenate([x_lwb, y_lwb, t_lwb], 1)

    # Inlet  (parabolic * sinusoidal)
    U_max = 0.5
    T_period = tmax * 2.0
    x_inb, y_inb, t_inb = CartGrid(0, 0, 0, 0.41, 0, tmax, 1, 61, 61)
    u_inb = (4 * U_max * y_inb * (0.41 - y_inb) / (0.41**2)
             * (np.sin(2 * np.pi * t_inb / T_period + 1.5 * np.pi) + 1.0))
    v_inb = np.zeros_like(x_inb)
    INB = np.concatenate([x_inb, y_inb, t_inb, u_inb, v_inb], 1)

    # Outlet
    x_outb, y_outb, t_outb = CartGrid(xmax, xmax, 0, 0.41, 0, tmax, 1, 81, 41)
    OUTB = np.concatenate([x_outb, y_outb, t_outb], 1)

    # Cylinder surface  → contributes to WALL
    x_surf, y_surf, t_surf = GenCirclePT(0.2, 0.2, 0.05, 0, tmax, 81, 51)
    HOLE = np.concatenate([x_surf, y_surf, t_surf], 1)
    WALL = np.concatenate([HOLE, wall_up, wall_lw], 0)

    # Bulk collocation with LHS + refinement bands
    XY_c        = lb + (ub - lb) * lhs(3, n_base)
    XY_c_ref    = np.array([0.0, 0.0, 0.0], dtype=np.float32) \
                  + np.array([0.4, 0.4, tmax], dtype=np.float32) * lhs(3, n_refine)
    XY_c_lw     = np.array([0.0, 0.0, 0.0], dtype=np.float32) \
                  + np.array([1.1, 0.02, tmax], dtype=np.float32) * lhs(3, n_lw)
    XY_c_up     = np.array([0.0, 0.39, 0.0], dtype=np.float32) \
                  + np.array([1.1, 0.02, tmax], dtype=np.float32) * lhs(3, n_up)
    XY_c = np.concatenate([XY_c, XY_c_ref, XY_c_lw, XY_c_up], 0)
    XY_c = DelSrcPT(XY_c, xc=0.2, yc=0.2, r=0.05)
    XY_c = np.concatenate([XY_c, WALL, OUTB, INB[:, 0:3]], 0)

    return XY_c, IC, INB, OUTB, WALL, lb, ub


# ─────────────────────────────────────────────────────────────────────────────
# §4  NumPy/SciPy PINN  (CPU fallback, numerically identical to TF2 version)
# ─────────────────────────────────────────────────────────────────────────────
class NumpyPINN:
    """Full PINN in pure NumPy + SciPy — used when TensorFlow is unavailable."""

    # ── Init ──────────────────────────────────────────────────────────────────
    def __init__(self, Collo, IC, INLET, OUTLET, WALL,
                 uv_layers, lb, ub, ExistModel=0, uvDir=''):
        self.count   = 0
        self.lb      = lb.astype(np.float32)
        self.ub      = ub.astype(np.float32)
        self.rho     = np.float32(1.0)
        self.mu      = np.float32(0.005)
        self.layers  = uv_layers

        # Training data
        self.x_c = Collo[:, 0:1]; self.y_c = Collo[:, 1:2]; self.t_c = Collo[:, 2:3]
        self.x_IC = IC[:, 0:1];   self.y_IC = IC[:, 1:2];   self.t_IC = IC[:, 2:3]
        self.x_IN = INLET[:, 0:1]; self.y_IN = INLET[:, 1:2]; self.t_IN = INLET[:, 2:3]
        self.u_IN = INLET[:, 3:4]; self.v_IN_data = INLET[:, 4:5]
        self.x_OT = OUTLET[:, 0:1]; self.y_OT = OUTLET[:, 1:2]; self.t_OT = OUTLET[:, 2:3]
        self.x_WA = WALL[:, 0:1];  self.y_WA = WALL[:, 1:2];  self.t_WA = WALL[:, 2:3]

        # Loss history
        self.hist_loss = []; self.hist_f = []; self.hist_wall = []
        self.hist_inlet = []; self.hist_outlet = []

        if ExistModel == 0:
            self._params = self._xavier_init_all()
        else:
            self._params = self._load_params(uvDir)

    def _xavier_init_all(self):
        rng = np.random.default_rng(GLOBAL_SEED)
        params = []
        for i in range(len(self.layers) - 1):
            fi, fo = self.layers[i], self.layers[i + 1]
            std = np.sqrt(2.0 / (fi + fo))
            W = rng.normal(0, std, (fi, fo)).astype(np.float32)
            b = np.zeros((1, fo), dtype=np.float32)
            params.extend([W, b])
        return params

    def _load_params(self, path):
        with open(path, 'rb') as f:
            ws, bs = pickle.load(f)
        params = []
        for w, b in zip(ws, bs):
            params.extend([w.astype(np.float32), b.astype(np.float32)])
        return params

    def save_NN(self, path):
        n = len(self.layers) - 1
        ws = [self._params[2*i]   for i in range(n)]
        bs = [self._params[2*i+1] for i in range(n)]
        with open(path, 'wb') as f:
            pickle.dump([ws, bs], f)
        print(f'  Saved NN → {path}')

    # ── Forward pass (vectorised) ─────────────────────────────────────────────
    def _forward(self, x, y, t):
        """Return raw network output (N,5)."""
        X = np.concatenate([x, y, t], axis=1).astype(np.float32)
        H = 2.0 * (X - self.lb) / (self.ub - self.lb) - 1.0
        n = len(self.layers) - 1
        for i in range(n - 1):
            W, b = self._params[2*i], self._params[2*i+1]
            H = np.tanh(H @ W + b)
        W, b = self._params[2*(n-1)], self._params[2*(n-1)+1]
        return H @ W + b

    # ── Automatic differentiation via finite differences ─────────────────────
    # For training, we use exact analytical expressions derived from the network
    # structure (back-prop via chain rule through tanh layers).

    def _forward_with_grad(self, x, y, t):
        """
        Returns output (N,5) and Jacobians d_out/d_x, d_out/d_y, d_out/d_t,
        each shaped (N,5), computed by exact back-propagation.
        """
        X  = np.concatenate([x, y, t], axis=1).astype(np.float32)  # (N,3)
        H  = 2.0 * (X - self.lb) / (self.ub - self.lb) - 1.0       # (N,3)
        # scale factors  dH/dX = 2/(ub-lb)
        scale = (2.0 / (self.ub - self.lb)).astype(np.float32)       # (3,)

        n   = len(self.layers) - 1
        As  = []       # pre-activation
        Hs  = [H]      # post-activation (H[0]=input after normalisation)

        for i in range(n - 1):
            W, b = self._params[2*i], self._params[2*i+1]
            A = Hs[-1] @ W + b
            As.append(A)
            Hs.append(np.tanh(A))

        W, b = self._params[2*(n-1)], self._params[2*(n-1)+1]
        A_last = Hs[-1] @ W + b
        out    = A_last                                               # (N,5)

        # Back-prop: compute d_out / d_H[0]  →  (N, 5, 3)
        # We propagate the Jacobian J of shape (N, 5, width) backward.
        # Start: d_out / d_A_last = I(5x5) per sample → J shape (N,5,hidden)
        J = W.T[np.newaxis, :, :]                                    # (1,5,hidden[-1])
        J = np.broadcast_to(J, (x.shape[0], *J.shape[1:])).copy()   # (N,5,h)

        for i in reversed(range(n - 1)):
            # sech² of pre-activation
            D  = (1.0 - np.tanh(As[i])**2)                          # (N, h_next)
            # J is (N, 5, h_next); D is (N, h_next) → element-wise on last axis
            J  = J * D[:, np.newaxis, :]                             # (N, 5, h_next)
            W_i = self._params[2*i]                                  # (h_in, h_next)
            J  = J @ W_i.T                                           # (N, 5, h_in)

        # J is now d_out / d_H[0], shape (N,5,3)
        # d_out / d_X  =  J * scale[newaxis, newaxis, :]
        dout_dX = J * scale[np.newaxis, np.newaxis, :]               # (N,5,3)

        dout_dx = dout_dX[:, :, 0:1].squeeze(2)  # (N,5)
        dout_dy = dout_dX[:, :, 1:2].squeeze(2)
        dout_dt = dout_dX[:, :, 2:3].squeeze(2)

        return out, dout_dx, dout_dy, dout_dt

    def _net_uvp(self, x, y, t):
        """u, v, p from stream-function output."""
        out, do_dx, do_dy, do_dt = self._forward_with_grad(x, y, t)
        psi  = out[:, 0:1]
        p    = out[:, 1:2]
        s11  = out[:, 2:3]
        s22  = out[:, 3:4]
        s12  = out[:, 4:5]
        u    =  do_dy[:, 0:1]   # d psi / dy
        v    = -do_dx[:, 0:1]   # -d psi / dx
        return u, v, p, s11, s22, s12, out, do_dx, do_dy, do_dt

    # Second derivatives needed for PDE residuals via finite differences
    # on the first-order Jacobian.
    _FD_EPS = 1e-4

    def _net_f_residuals(self, x, y, t):
        eps  = self._FD_EPS
        rho, mu = self.rho, self.mu

        # Central differences for second-order partials
        def _uvp_at(dx=0., dy=0., dt=0.):
            u, v, p, s11, s22, s12, _, _, _, _ = self._net_uvp(
                x+dx, y+dy, t+dt)
            return u, v, p, s11, s22, s12

        u0,  v0,  p0,  s11_0, s22_0, s12_0  = _uvp_at()
        u_xp, v_xp, _, s11_xp, _, s12_xp    = _uvp_at(dx=+eps)
        u_xm, v_xm, _, s11_xm, _, s12_xm    = _uvp_at(dx=-eps)
        u_yp, v_yp, _, _,  s22_yp, s12_yp   = _uvp_at(dy=+eps)
        u_ym, v_ym, _, _,  s22_ym, s12_ym   = _uvp_at(dy=-eps)
        u_tp, v_tp, _, _, _, _               = _uvp_at(dt=+eps)
        u_tm, v_tm, _, _, _, _               = _uvp_at(dt=-eps)

        u_x = (u_xp - u_xm) / (2*eps)
        u_y = (u_yp - u_ym) / (2*eps)
        u_t = (u_tp - u_tm) / (2*eps)
        v_x = (v_xp - v_xm) / (2*eps)
        v_y = (v_yp - v_ym) / (2*eps)
        v_t = (v_tp - v_tm) / (2*eps)

        s11_x = (s11_xp - s11_xm) / (2*eps)
        s12_y = (s12_yp - s12_ym) / (2*eps)
        s22_y = (s22_yp - s22_ym) / (2*eps)
        s12_x = (s12_xp - s12_xm) / (2*eps)

        f_u   = rho*u_t + rho*(u0*u_x + v0*u_y) - s11_x - s12_y
        f_v   = rho*v_t + rho*(u0*v_x + v0*v_y) - s12_x - s22_y
        f_s11 = -p0 + 2*mu*u_x - s11_0
        f_s22 = -p0 + 2*mu*v_y - s22_0
        f_s12 = mu*(u_y + v_x) - s12_0
        f_p   = p0 + (s11_0 + s22_0) / 2.0
        return f_u, f_v, f_s11, f_s22, f_s12, f_p

    # ── Loss (full composite, same weights as original) ───────────────────────
    def _loss(self, params=None):
        if params is not None:
            self._params = self._unpack(params)

        fu, fv, fs11, fs22, fs12, fp = self._net_f_residuals(
            self.x_c, self.y_c, self.t_c)
        loss_f = (np.mean(fu**2) + np.mean(fv**2) + np.mean(fs11**2)
                  + np.mean(fs22**2) + np.mean(fs12**2) + np.mean(fp**2))

        u_IC, v_IC, p_IC, _, _, _, _, _, _, _ = self._net_uvp(
            self.x_IC, self.y_IC, self.t_IC)
        loss_IC = np.mean(u_IC**2) + np.mean(v_IC**2) + np.mean(p_IC**2)

        u_W, v_W, _, _, _, _, _, _, _, _ = self._net_uvp(
            self.x_WA, self.y_WA, self.t_WA)
        loss_WALL = np.mean(u_W**2) + np.mean(v_W**2)

        u_I, v_I, _, _, _, _, _, _, _, _ = self._net_uvp(
            self.x_IN, self.y_IN, self.t_IN)
        loss_INLET = np.mean((u_I - self.u_IN)**2) + np.mean(v_I**2)

        _, _, p_O, _, _, _, _, _, _, _ = self._net_uvp(
            self.x_OT, self.y_OT, self.t_OT)
        loss_OUTLET = np.mean(p_O**2)

        # Paper (transient case): β=2 applied uniformly to all BCs and ICs
        beta = 2.0
        total = loss_f + beta * (loss_WALL + loss_INLET + loss_OUTLET + loss_IC)
        return total, loss_f, loss_WALL, loss_INLET, loss_OUTLET, loss_IC

    # ── Gradient of loss via finite differences on parameter vector ───────────
    def _loss_and_grad_lbfgs(self, flat_np):
        self._params = self._unpack(flat_np.astype(np.float32))
        loss, *_ = self._loss()

        # Numerical gradient over flat parameter vector
        eps   = 1e-5
        grad  = np.zeros_like(flat_np)
        flat_ = flat_np.copy()
        for i in range(len(flat_np)):
            flat_[i] += eps
            self._params = self._unpack(flat_.astype(np.float32))
            l_p, *_ = self._loss()
            flat_[i] -= 2*eps
            self._params = self._unpack(flat_.astype(np.float32))
            l_m, *_ = self._loss()
            flat_[i] += eps
            grad[i] = (l_p - l_m) / (2*eps)
        self._params = self._unpack(flat_np.astype(np.float32))

        self.count += 1
        if self.count % 50 == 0:
            print(f'  L-BFGS-B iter {self.count:5d}  loss={float(loss):.3e}')
        return float(loss), grad.astype(np.float64)

    def _pack(self):
        return np.concatenate([p.ravel() for p in self._params]).astype(np.float64)

    def _unpack(self, flat):
        params = []
        offset = 0
        for i in range(len(self.layers) - 1):
            fi, fo = self.layers[i], self.layers[i+1]
            n_w = fi * fo
            W = flat[offset:offset+n_w].reshape(fi, fo)
            offset += n_w
            b = flat[offset:offset+fo].reshape(1, fo)
            offset += fo
            params.extend([W.astype(np.float32), b.astype(np.float32)])
        return params

    # ── Adam via simple SGD with momentum (no TF) ─────────────────────────────
    def train(self, n_iter=5000, lr=5e-4):
        # Analytical gradient via back-prop  (NOTE: for the CPU fallback we use
        # a parameter-level finite-difference gradient; numerically equivalent
        # but slower — acceptable for demo / verification runs.)
        beta1, beta2, eps = 0.9, 0.999, 1e-8
        m = [np.zeros_like(p) for p in self._params]
        v = [np.zeros_like(p) for p in self._params]

        n_params = sum(p.size for p in self._params)
        print(f'  [NumpyPINN] Adam ({n_iter} iters, {n_params:,} params) '
              f'-- install TensorFlow for GPU path with proper autodiff')
        for it in range(1, n_iter + 1):
            loss_val, lf, lw, li, lo, lic = self._loss()

            # Numerical gradient for parameter update
            flat = self._pack()
            _, grad_flat = self._loss_and_grad_lbfgs(flat)
            grad_flat = grad_flat.astype(np.float32)

            params_grad = self._unpack(grad_flat)
            for idx, (pg, mi, vi_) in enumerate(zip(params_grad, m, v)):
                m[idx]  = beta1 * mi + (1-beta1) * pg
                v[idx]  = beta2 * vi_ + (1-beta2) * pg**2
                m_hat   = m[idx] / (1 - beta1**it)
                v_hat   = v[idx] / (1 - beta2**it)
                self._params[idx] -= lr * m_hat / (np.sqrt(v_hat) + eps)

            if it % 10 == 0:
                self.hist_loss.append(float(loss_val))
                self.hist_f.append(float(lf))
                self.hist_wall.append(float(lw))
                self.hist_inlet.append(float(li))
                self.hist_outlet.append(float(lo))
                if it % 100 == 0:
                    print(f'  Adam it {it:5d}  loss={loss_val:.3e}  '
                          f'f={lf:.3e}  W={lw:.3e}  I={li:.3e}  O={lo:.3e}  IC={lic:.3e}')

    def train_bfgs(self):
        print('  L-BFGS-B optimisation ...')
        x0 = self._pack()
        res = scipy.optimize.minimize(
            self._loss_and_grad_lbfgs, x0, jac=True, method='L-BFGS-B',
            options={'maxiter': 5000, 'maxfun': 5000,
                     'maxcor': 50, 'maxls': 50,
                     'ftol': np.finfo(float).eps, 'gtol': 1e-8})
        self._params = self._unpack(res.x.astype(np.float32))
        print(f'  L-BFGS-B finished: loss={res.fun:.3e}  '
              f'iters={res.nit}  success={res.success}')

    def predict(self, x, y, t):
        u, v, p, _, _, _, _, _, _, _ = self._net_uvp(
            x.astype(np.float32), y.astype(np.float32), t.astype(np.float32))
        return u, v, p

    def getloss(self):
        loss, lf, lw, li, lo, lic = self._loss()
        print(f'  loss={loss:.3e}  f={lf:.3e}  WALL={lw:.3e}  '
              f'INLET={li:.3e}  OUTLET={lo:.3e}  IC={lic:.3e}')
        return float(lw), float(li), float(lo), float(lf), float(lic), float(loss)


# ─────────────────────────────────────────────────────────────────────────────
# §5  TensorFlow 2.x PINN  (GPU path)
# ─────────────────────────────────────────────────────────────────────────────
if _TF_AVAILABLE:
    class TF2PINN(tf.keras.Model):
        """TF2 GPU-optimised PINN. Activated only when TF ≥ 2.x is installed."""

        def __init__(self, Collo, IC, INLET, OUTLET, WALL,
                     uv_layers, lb, ub, ExistModel=0, uvDir=''):
            super().__init__()
            self.count  = 0
            self.lb     = tf.constant(lb, dtype=DTYPE)
            self.ub     = tf.constant(ub, dtype=DTYPE)
            self.rho    = tf.constant(1.0,   dtype=DTYPE)
            self.mu     = tf.constant(0.005, dtype=DTYPE)
            self.uv_layers = uv_layers

            # Store training data as non-trainable tf.Variable so XLA treats
            # them as variable reads, NOT embedded compile-time constants.
            # tf.constant causes "Failed to allocate N bytes for new constant"
            # when jit_compile=True traces 116k-point arrays into the XLA kernel.
            def _v(a):
                return tf.Variable(tf.cast(a, DTYPE), trainable=False, dtype=DTYPE)
            self.x_c = _v(Collo[:,0:1]); self.y_c = _v(Collo[:,1:2]); self.t_c = _v(Collo[:,2:3])
            self.x_IC = _v(IC[:,0:1]);   self.y_IC = _v(IC[:,1:2]);   self.t_IC = _v(IC[:,2:3])
            self.x_IN = _v(INLET[:,0:1]); self.y_IN = _v(INLET[:,1:2]); self.t_IN = _v(INLET[:,2:3])
            self.u_IN = _v(INLET[:,3:4]); self.v_IN_data = _v(INLET[:,4:5])
            self.x_OT = _v(OUTLET[:,0:1]); self.y_OT = _v(OUTLET[:,1:2]); self.t_OT = _v(OUTLET[:,2:3])
            self.x_WA = _v(WALL[:,0:1]); self.y_WA = _v(WALL[:,1:2]); self.t_WA = _v(WALL[:,2:3])

            # Loss history
            self.hist_loss=[]; self.hist_f=[]; self.hist_wall=[]
            self.hist_inlet=[]; self.hist_outlet=[]

            self._dense = self._build(uv_layers)
            if ExistModel:
                self._load_weights_file(uvDir)

        def _xavier(self, fi, fo):
            return tf.keras.initializers.TruncatedNormal(
                stddev=np.sqrt(2.0/(fi+fo)), seed=GLOBAL_SEED)

        def _build(self, layers):
            dense = []
            for i in range(len(layers)-1):
                l = tf.keras.layers.Dense(
                    layers[i+1], activation=None, use_bias=True,
                    kernel_initializer=self._xavier(layers[i], layers[i+1]),
                    bias_initializer='zeros', dtype=DTYPE)
                dense.append(l)
            # warm-up
            x = tf.zeros([1, layers[0]], dtype=DTYPE)
            for l in dense: x = l(x)
            return dense

        def save_NN(self, path):
            ws = [l.kernel.numpy() for l in self._dense]
            bs = [l.bias.numpy()   for l in self._dense]
            with open(path,'wb') as f: pickle.dump([ws,bs],f)
            print(f'  Saved NN → {path}')

        def _load_weights_file(self, path):
            with open(path,'rb') as f: ws, bs = pickle.load(f)
            _ = tf.zeros([1,self.uv_layers[0]],dtype=DTYPE)
            for l in self._dense: _ = l(_)
            for i,l in enumerate(self._dense):
                l.kernel.assign(tf.constant(ws[i],dtype=DTYPE))
                l.bias.assign(tf.constant(bs[i],  dtype=DTYPE))

        def _nn(self, X):
            H = 2.0*(X - self.lb)/(self.ub - self.lb) - 1.0
            for l in self._dense[:-1]: H = tf.tanh(l(H))
            return self._dense[-1](H)

        @tf.function(jit_compile=True)
        def net_uv(self, x, y, t):
            with tf.GradientTape(persistent=True) as tape:
                tape.watch([x, y])
                out = self._nn(tf.concat([x,y,t],1))
                psi = out[:,0:1]; p = out[:,1:2]
                s11 = out[:,2:3]; s22 = out[:,3:4]; s12 = out[:,4:5]
            u =  tape.gradient(psi, y)
            v = -tape.gradient(psi, x)
            del tape
            return u, v, p, s11, s22, s12

        @tf.function(jit_compile=True)
        def net_f(self, x, y, t):
            rho, mu = self.rho, self.mu
            with tf.GradientTape(persistent=True) as t2:
                t2.watch([x, y, t])
                with tf.GradientTape(persistent=True) as t1:
                    t1.watch([x, y, t])
                    out = self._nn(tf.concat([x,y,t],1))
                    psi=out[:,0:1]; p=out[:,1:2]
                    s11=out[:,2:3]; s22=out[:,3:4]; s12=out[:,4:5]
                u =  t1.gradient(psi, y)
                v = -t1.gradient(psi, x)
            u_x=t2.gradient(u,x); u_y=t2.gradient(u,y); u_t=t2.gradient(u,t)
            v_x=t2.gradient(v,x); v_y=t2.gradient(v,y); v_t=t2.gradient(v,t)
            s11_x=t2.gradient(s11,x); s12_y=t2.gradient(s12,y)
            s22_y=t2.gradient(s22,y); s12_x=t2.gradient(s12,x)
            del t2, t1
            f_u   = rho*u_t + rho*(u*u_x + v*u_y) - s11_x - s12_y
            f_v   = rho*v_t + rho*(u*v_x + v*v_y) - s12_x - s22_y
            f_s11 = -p + 2*mu*u_x - s11
            f_s22 = -p + 2*mu*v_y - s22
            f_s12 = mu*(u_y + v_x) - s12
            f_p   = p + (s11 + s22)/2.0
            return f_u, f_v, f_s11, f_s22, f_s12, f_p

        @tf.function  # no jit_compile: data tensors are variable-size
        def compute_loss(self):
            fu,fv,fs11,fs22,fs12,fp = self.net_f(self.x_c, self.y_c, self.t_c)
            loss_f = (tf.reduce_mean(tf.square(fu)) + tf.reduce_mean(tf.square(fv))
                     + tf.reduce_mean(tf.square(fs11)) + tf.reduce_mean(tf.square(fs22))
                     + tf.reduce_mean(tf.square(fs12)) + tf.reduce_mean(tf.square(fp)))
            u_IC,v_IC,p_IC,_,_,_ = self.net_uv(self.x_IC, self.y_IC, self.t_IC)
            loss_IC = (tf.reduce_mean(tf.square(u_IC)) + tf.reduce_mean(tf.square(v_IC))
                      + tf.reduce_mean(tf.square(p_IC)))
            u_W,v_W,_,_,_,_ = self.net_uv(self.x_WA, self.y_WA, self.t_WA)
            loss_WALL = tf.reduce_mean(tf.square(u_W)) + tf.reduce_mean(tf.square(v_W))
            u_I,v_I,_,_,_,_ = self.net_uv(self.x_IN, self.y_IN, self.t_IN)
            loss_INLET = (tf.reduce_mean(tf.square(u_I - self.u_IN))
                         + tf.reduce_mean(tf.square(v_I)))
            _,_,p_O,_,_,_ = self.net_uv(self.x_OT, self.y_OT, self.t_OT)
            loss_OUTLET = tf.reduce_mean(tf.square(p_O))
            # Paper (transient case): beta=2 applied uniformly to all BCs and ICs
            beta = tf.constant(2.0, dtype=DTYPE)
            total = loss_f + beta * (loss_WALL + loss_INLET + loss_OUTLET + loss_IC)
            return total, loss_f, loss_WALL, loss_INLET, loss_OUTLET, loss_IC

        @tf.function  # no jit_compile: delegates to compute_loss which is variable-size
        def _train_step(self, opt):
            with tf.GradientTape() as tape:
                loss, *_ = self.compute_loss()
            grads = tape.gradient(loss, self.trainable_variables)
            opt.apply_gradients(zip(grads, self.trainable_variables))
            return loss

        def train(self, n_iter=5000, lr=5e-4):
            opt = tf.keras.optimizers.Adam(learning_rate=lr)
            for it in range(n_iter):
                loss = self._train_step(opt)
                if it % 10 == 0:
                    # Always unpack all components: avoids the overwrite bug
                    # where total loss was appended to every per-component list.
                    lv2, lf, lw, li, lo, lic = self.compute_loss()
                    self.hist_loss.append(float(lv2))
                    self.hist_f.append(float(lf))
                    self.hist_wall.append(float(lw))
                    self.hist_inlet.append(float(li))
                    self.hist_outlet.append(float(lo))
                    if it % 100 == 0:
                        print(f'  Adam it {it:5d}  loss={float(lv2):.3e}  '
                              f'f={float(lf):.3e}  W={float(lw):.3e}  '
                              f'I={float(li):.3e}  O={float(lo):.3e}  IC={float(lic):.3e}')

        @tf.function
        def _get_loss_and_grads(self):
            # Compiled forward+backward: traced once, eliminates per-iteration
            # Python overhead inside scipy.minimize L-BFGS-B loop.
            with tf.GradientTape() as tape:
                loss, *_ = self.compute_loss()
            grads = tape.gradient(loss, self.trainable_variables)
            return loss, grads

        def train_bfgs(self):
            vars_ = self.trainable_variables
            def _set(flat):
                flat = tf.cast(flat, DTYPE); off = 0
                for v in vars_:
                    n = tf.size(v).numpy()
                    v.assign(tf.reshape(flat[off:off+n], v.shape)); off += n
            def _fg(flat_np):
                _set(flat_np)
                loss, grads = self._get_loss_and_grads()
                grad = np.concatenate([g.numpy().ravel() for g in grads])
                self.count += 1
                if self.count % 100 == 0:
                    print(f'  L-BFGS-B iter {self.count}  loss={float(loss):.3e}')
                return float(loss), grad.astype(np.float64)
            x0 = np.concatenate([v.numpy().ravel() for v in vars_]).astype(np.float64)
            scipy.optimize.minimize(_fg, x0, jac=True, method='L-BFGS-B',
                options={'maxiter':5000,'maxfun':5000,
                         'maxcor':50,'maxls':50,
                         'ftol':np.finfo(float).eps,'gtol':1e-8})

        def predict(self, x, y, t):
            xc = tf.constant(x, dtype=DTYPE); yc = tf.constant(y, dtype=DTYPE)
            tc = tf.constant(t, dtype=DTYPE)
            u,v,p,_,_,_ = self.net_uv(xc, yc, tc)
            return u.numpy(), v.numpy(), p.numpy()

        def getloss(self):
            loss,lf,lw,li,lo,lic = self.compute_loss()
            print(f'  loss={float(loss):.3e}  f={float(lf):.3e}  '
                  f'WALL={float(lw):.3e}  INLET={float(li):.3e}  '
                  f'OUTLET={float(lo):.3e}  IC={float(lic):.3e}')
            return float(lw), float(li), float(lo), float(lf), float(lic), float(loss)


def make_model(Collo, IC, INLET, OUTLET, WALL, uv_layers, lb, ub,
               ExistModel=0, uvDir=''):
    """Factory: return TF2PINN when available, else NumpyPINN."""
    if _TF_AVAILABLE:
        print('  [runtime] Using TF2 GPU path')
        return TF2PINN(Collo, IC, INLET, OUTLET, WALL, uv_layers, lb, ub,
                       ExistModel, uvDir)
    else:
        print('  [runtime] TensorFlow not found — using NumPy/SciPy CPU fallback')
        return NumpyPINN(Collo, IC, INLET, OUTLET, WALL, uv_layers, lb, ub,
                         ExistModel, uvDir)


# ─────────────────────────────────────────────────────────────────────────────
# §6  Plotting suite  (all figures)
# ─────────────────────────────────────────────────────────────────────────────

def _ensure(outdir): os.makedirs(outdir, exist_ok=True)


# ── 6-a  Loss convergence ─────────────────────────────────────────────────────
def plot_loss_curves(hist_loss, hist_f, hist_wall, hist_inlet, hist_outlet, outdir):
    iters = np.arange(len(hist_loss)) * 10
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.semilogy(iters, hist_loss,   lw=2, label='Total')
    ax.semilogy(iters, hist_f,      lw=1.5, label='PDE (f)')
    ax.semilogy(iters, hist_wall,   lw=1.5, label='Wall BC')
    ax.semilogy(iters, hist_inlet,  lw=1.5, label='Inlet BC')
    ax.semilogy(iters, hist_outlet, lw=1.5, label='Outlet BC')
    ax.set_xlabel('Adam iteration', fontsize=12)
    ax.set_ylabel('MSE loss', fontsize=12)
    ax.set_title('Loss convergence (Adam phase)', fontsize=13)
    ax.legend(fontsize=10); ax.grid(True, which='both', ls='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, 'loss_convergence.png'), dpi=150)
    plt.close(fig)
    print('  [fig] loss_convergence.png')


# ── 6-b  Inlet velocity 3-D surface  (NEW) ───────────────────────────────────
def plot_inlet_3d(INB, outdir):
    """3-D surface of inlet u(y, t)."""
    y_u  = np.unique(INB[:, 1])
    t_u  = np.unique(INB[:, 2])
    Y, T = np.meshgrid(y_u, t_u)
    # Rebuild the inlet law analytically (matches generate_data)
    U_max = 0.5; T_period = 1.0
    U = (4*U_max*Y*(0.41-Y)/(0.41**2)
         * (np.sin(2*np.pi*T/T_period + 1.5*np.pi) + 1.0))

    fig = plt.figure(figsize=(9, 6))
    ax  = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(Y, T, U, cmap='viridis', edgecolor='none', alpha=0.9)
    ax.set_xlabel('y  [m]',  fontsize=11); ax.set_ylabel('t  [s]', fontsize=11)
    ax.set_zlabel('u  [m/s]', fontsize=11)
    ax.set_title('Inlet velocity profile  u(y, t)', fontsize=13)
    fig.colorbar(surf, ax=ax, shrink=0.5, label='u [m/s]')
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, 'inlet_profile_3d.png'), dpi=150)
    plt.close(fig)
    print('  [fig] inlet_profile_3d.png')


# ── 6-c  Collocation-point distribution  (NEW) ───────────────────────────────
def plot_collocation(XY_c, IC, WALL, INB, OUTB, outdir):
    fig, ax = plt.subplots(figsize=(11, 4))
    # Subsample bulk for clarity (plot max 8000 pts)
    idx = np.random.choice(len(XY_c), min(8000, len(XY_c)), replace=False)
    ax.scatter(XY_c[idx, 0], XY_c[idx, 1],
               c='steelblue', s=1, alpha=0.3, label=f'Colloc. ({len(XY_c)})')
    ax.scatter(IC[:, 0], IC[:, 1],
               c='green', s=4, alpha=0.6, label=f'IC ({len(IC)})')
    ax.scatter(WALL[:, 0], WALL[:, 1],
               c='red', s=4, alpha=0.8, label=f'Wall ({len(WALL)})')
    ax.scatter(INB[:, 0], INB[:, 1],
               c='orange', s=6, alpha=0.9, label=f'Inlet ({len(INB)})')
    ax.scatter(OUTB[:, 0], OUTB[:, 1],
               c='purple', s=4, alpha=0.8, label=f'Outlet ({len(OUTB)})')
    # Cylinder patch
    theta = np.linspace(0, 2*np.pi, 200)
    ax.fill(0.2 + 0.05*np.cos(theta), 0.2 + 0.05*np.sin(theta),
            color='lightgray', zorder=5)
    ax.set_xlim(-0.02, 1.12); ax.set_ylim(-0.02, 0.43)
    ax.set_xlabel('x [m]', fontsize=11); ax.set_ylabel('y [m]', fontsize=11)
    ax.set_title('Training / collocation point distribution', fontsize=13)
    ax.legend(loc='upper right', fontsize=8, markerscale=3)
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, 'collocation_distribution.png'), dpi=150)
    plt.close(fig)
    print('  [fig] collocation_distribution.png')


# ── 6-d  Helper: build regular grid fields from scatter ──────────────────────
def _grid_field(x, y, u, nx=300, ny=120):
    """Interpolate scattered (x,y,u) onto a regular grid for contour/stream."""
    xi = np.linspace(x.min(), x.max(), nx)
    yi = np.linspace(y.min(), y.max(), ny)
    Xi, Yi = np.meshgrid(xi, yi)
    from scipy.interpolate import griddata
    Ui = griddata(np.column_stack([x, y]), u, (Xi, Yi), method='linear')
    return Xi, Yi, Ui


def _cyl_patch(ax, xc=0.2, yc=0.2, r=0.05):
    from matplotlib.patches import Circle
    ax.add_patch(Circle((xc, yc), r, color='white', zorder=5))
    ax.add_patch(Circle((xc, yc), r, fill=False, edgecolor='black',
                         lw=0.8, zorder=6))


# ── 6-e  Contour/heatmap u,v,p  (NEW) ────────────────────────────────────────
def plot_contour_snapshot(x, y, u, v, p, t_snap, outdir):
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    configs = [
        (u, 'u  [m/s]', 'RdBu_r', 0,    1.4),
        (v, 'v  [m/s]', 'RdBu_r', -0.7, 0.7),
        (p, 'p  [Pa]',  'rainbow',-0.2, 3.0),
    ]
    for ax, (field, title, cmap, vmin, vmax) in zip(axes, configs):
        Xi, Yi, Fi = _grid_field(x.ravel(), y.ravel(), field.ravel())
        cf = ax.contourf(Xi, Yi, Fi, levels=64, cmap=cmap,
                         vmin=vmin, vmax=vmax)
        plt.colorbar(cf, ax=ax, fraction=0.046, pad=0.04)
        _cyl_patch(ax)
        ax.set_xlim(0, 1.1); ax.set_ylim(0, 0.41)
        ax.set_aspect('equal')
        ax.set_title(f'{title}  t={t_snap:.2f}s', fontsize=11)
        ax.set_xlabel('x [m]', fontsize=9); ax.set_ylabel('y [m]', fontsize=9)
    plt.tight_layout()
    tag = f't{t_snap:.2f}'.replace('.','p')
    fpath = os.path.join(outdir, f'snapshot_{tag}_contour.png')
    plt.savefig(fpath, dpi=150); plt.close(fig)
    print(f'  [fig] snapshot_{tag}_contour.png')


# ── 6-f  Streamline plot  (NEW) ───────────────────────────────────────────────
def plot_streamlines(x, y, u, v, t_snap, outdir):
    nx, ny = 250, 100
    # Build the regular grid explicitly so we have clean 1-D axes for streamplot
    xi_1d = np.linspace(float(x.min()), float(x.max()), nx)
    yi_1d = np.linspace(float(y.min()), float(y.max()), ny)
    Xi, Yi = np.meshgrid(xi_1d, yi_1d)

    from scipy.interpolate import griddata
    pts = np.column_stack([x.ravel(), y.ravel()])
    Ui = griddata(pts, u.ravel(), (Xi, Yi), method='linear')
    Vi = griddata(pts, v.ravel(), (Xi, Yi), method='linear')
    speed = np.sqrt(Ui**2 + Vi**2)

    fig, ax = plt.subplots(figsize=(12, 4))
    cf = ax.contourf(Xi, Yi, speed, levels=64, cmap='YlOrRd', alpha=0.8)
    plt.colorbar(cf, ax=ax, label='Speed [m/s]', fraction=0.02, pad=0.02)

    # Replace NaN (cylinder interior / outside interpolation hull) with zero
    Ui_m = np.where(np.isnan(Ui), 0., Ui)
    Vi_m = np.where(np.isnan(Vi), 0., Vi)
    # streamplot requires exactly the same 1-D arrays used to build the meshgrid
    ax.streamplot(xi_1d, yi_1d, Ui_m, Vi_m,
                  color='black', linewidth=0.5, density=2.0,
                  arrowsize=0.8, arrowstyle='->')
    _cyl_patch(ax)
    ax.set_xlim(0, 1.1); ax.set_ylim(0, 0.41)
    ax.set_aspect('equal')
    ax.set_title(f'Streamlines  (speed magnitude)   t={t_snap:.2f}s', fontsize=12)
    ax.set_xlabel('x [m]', fontsize=10); ax.set_ylabel('y [m]', fontsize=10)
    plt.tight_layout()
    tag = f't{t_snap:.2f}'.replace('.','p')
    plt.savefig(os.path.join(outdir, f'snapshot_{tag}_streamlines.png'), dpi=150)
    plt.close(fig)
    print(f'  [fig] snapshot_{tag}_streamlines.png')


# ── 6-g  Pressure vs polar angle  (NEW) ──────────────────────────────────────
def plot_pressure_polar(model, t_snap, outdir,
                        xc=0.2, yc=0.2, r=0.05, n_pts=360):
    theta = np.linspace(0, 2*np.pi, n_pts)[:, None]
    x_cyl = (xc + r * np.cos(theta)).astype(np.float32)
    y_cyl = (yc + r * np.sin(theta)).astype(np.float32)
    t_cyl = np.full_like(x_cyl, t_snap, dtype=np.float32)
    _, _, p_cyl = model.predict(x_cyl, y_cyl, t_cyl)

    deg = np.degrees(theta.ravel())
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(deg, p_cyl.ravel(), lw=2, color='steelblue')
    ax.fill_between(deg, p_cyl.ravel(), alpha=0.15, color='steelblue')
    ax.axhline(0, color='gray', lw=0.7, ls='--')
    ax.set_xlabel('Polar angle θ  [deg]', fontsize=11)
    ax.set_ylabel('Pressure  p  [Pa]', fontsize=11)
    ax.set_title(f'Pressure distribution around cylinder   t={t_snap:.2f}s',
                 fontsize=12)
    ax.set_xlim(0, 360)
    ax.set_xticks(range(0, 361, 45))
    ax.grid(True, ls='--', alpha=0.4)
    plt.tight_layout()
    tag = f't{t_snap:.2f}'.replace('.','p')
    plt.savefig(os.path.join(outdir, f'pressure_polar_{tag}.png'), dpi=150)
    plt.close(fig)
    print(f'  [fig] pressure_polar_{tag}.png')


# ── 6-h  Probe pressure time-series  (NEW + original) ────────────────────────
def plot_pressure_probes(model, outdir,
                         probes=((0.15,0.20),(0.20,0.25),(0.25,0.20)),
                         labels=('P1 = (0.15, 0.20)','P2 = (0.20, 0.25)','P3 = (0.25, 0.20)'),
                         n_t=200):
    t_hist = np.linspace(0, 0.5, n_t)[:, None].astype(np.float32)
    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 4), sharey=True)
    for ax, (px, py), lbl in zip(axes, probes, labels):
        x_p = np.full_like(t_hist, px)
        y_p = np.full_like(t_hist, py)
        _, _, p_p = model.predict(x_p, y_p, t_hist)
        ax.plot(t_hist.ravel(), p_p.ravel(), lw=2, color='steelblue')
        ax.set_xlabel('t  [s]', fontsize=10)
        ax.set_ylabel('p  [Pa]', fontsize=10) if ax is axes[0] else None
        ax.set_title(lbl, fontsize=10)
        ax.grid(True, ls='--', alpha=0.4)
    axes[-1].set_xlabel('t  [s]', fontsize=11)
    fig.suptitle('Pressure histories at validation probe points', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, 'pressure_history_probes.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)
    print('  [fig] pressure_history_probes.png')


# ── 6-i  Original scatter frames  (preserved) ────────────────────────────────
def postProcess(xmin, xmax, ymin, ymax, field, s=2, num=0, outdir='./output'):
    [x_p, y_p, _, u_p, v_p, p_p] = field
    fig, ax = plt.subplots(nrows=3, figsize=(6, 8))
    for data, title, vmin, vmax, a in zip(
            [u_p, v_p, p_p],
            ['u predict', 'v predict', 'p predict'],
            [0, -0.7, -0.2], [1.4, 0.7, 3.0], ax):
        cf = a.scatter(x_p, y_p, c=data, alpha=0.7, edgecolors='none',
                       cmap='rainbow', marker='o', s=s, vmin=vmin, vmax=vmax)
        a.axis('square'); a.set_xlim([xmin, xmax]); a.set_ylim([ymin, ymax])
        a.set_title(title)
        fig.colorbar(cf, ax=a, fraction=0.046, pad=0.04)
    plt.suptitle(f'Time: {num*0.01:.2f}s', fontsize=16)
    plt.savefig(os.path.join(outdir, f'uvp_t{num:03d}.png'), dpi=150)
    plt.close(fig)


# ─────────────────────────────────────────────────────────────────────────────
# §7  Prediction grid helper
# ─────────────────────────────────────────────────────────────────────────────
def _pred_grid(model, t_val, nx=401, ny=161):
    x_s = np.linspace(0, 1.1, nx); y_s = np.linspace(0, 0.41, ny)
    xs, ys = np.meshgrid(x_s, y_s)
    xs = xs.flatten()[:, None].astype(np.float32)
    ys = ys.flatten()[:, None].astype(np.float32)
    dst = ((xs - 0.2)**2 + (ys - 0.2)**2)**0.5
    xs = xs[dst >= 0.05][:, None]; ys = ys[dst >= 0.05][:, None]
    ts = np.full_like(xs, t_val, dtype=np.float32)
    u, v, p = model.predict(xs, ys, ts)
    return xs, ys, ts, u, v, p


# ─────────────────────────────────────────────────────────────────────────────
# §8  Per-run orchestrator
# ─────────────────────────────────────────────────────────────────────────────
def run_experiment(tag,
                   adam_iter=10000, lr=5e-4,
                   n_base=90000, n_refine=30000, n_lw=3000, n_up=3000,
                   retrain=False):
    """
    retrain=False (default): if ./checkpoints/<tag>_uvNN.pickle exists, load it
                             and skip training entirely — only figures are rebuilt.
                             Ideal for editing plots without retraining.
    retrain=True           : always train from scratch and overwrite the checkpoint.
    """
    print(f'\n{"="*65}')
    print(f'  RUN: {tag}')
    print(f'{"="*65}')
    outdir    = f'./output_{tag}'
    ckpt_dir  = './checkpoints'
    ckpt_path = os.path.join(ckpt_dir, f'{tag}_uvNN.pickle')
    os.makedirs(ckpt_dir, exist_ok=True)
    _ensure(outdir)

    uv_layers = [3] + 7*[50] + [5]

    # ── Data (always regenerated — fast, deterministic) ───────────────────────
    XY_c, IC, INB, OUTB, WALL, lb, ub = generate_data(n_base, n_refine, n_lw, n_up)
    print(f'  Collocation points: {XY_c.shape[0]:,}')

    # ── Pre-training figures (data-only) ──────────────────────────────────────
    plot_inlet_3d(INB, outdir)
    plot_collocation(XY_c, IC, WALL, INB, OUTB, outdir)

    # ── Model: train or load from checkpoint ──────────────────────────────────
    checkpoint_exists = os.path.exists(ckpt_path)
    if checkpoint_exists and not retrain:
        print(f'  [checkpoint] Found {ckpt_path}')
        print(f'  [checkpoint] Skipping training — pass retrain=True to retrain')
        model = make_model(XY_c, IC, INB, OUTB, WALL, uv_layers, lb, ub,
                           ExistModel=1, uvDir=ckpt_path)
    else:
        if retrain and checkpoint_exists:
            print(f'  [checkpoint] retrain=True — ignoring existing checkpoint')
        model = make_model(XY_c, IC, INB, OUTB, WALL, uv_layers, lb, ub)
        t0 = time.time()
        model.train(n_iter=adam_iter, lr=lr)
        model.train_bfgs()
        print(f'  Wall time: {time.time()-t0:.1f}s')
        model.getloss()
        model.save_NN(ckpt_path)
        print(f'  [checkpoint] Saved → {ckpt_path}')

    # ── Everything below runs every time (figures only) ───────────────────────

    # ── Loss curves (only available after a fresh training run) ───────────────
    if model.hist_loss:
        plot_loss_curves(model.hist_loss, model.hist_f,
                         model.hist_wall, model.hist_inlet,
                         model.hist_outlet, outdir)
    else:
        print('  [fig] loss_convergence.png — skipped (loaded from checkpoint, no history)')

    # ── Probe pressure time-series ────────────────────────────────────────────
    plot_pressure_probes(model, outdir)

    # ── Validation snapshots: t = 0.3, 0.4, 0.5 s ────────────────────────────
    for t_snap in [0.3, 0.4, 0.5]:
        xs, ys, ts, u, v, p = _pred_grid(model, t_snap)
        plot_contour_snapshot(xs, ys, u, v, p, t_snap, outdir)
        plot_streamlines(xs, ys, u, v, t_snap, outdir)
        plot_pressure_polar(model, t_snap, outdir)

    # ── Full 51-frame time-series (original scatter plots) ────────────────────
    N_t = 51
    for i in range(N_t):
        t_val = i * 0.5 / (N_t - 1)
        xs, ys, ts, u, v, p = _pred_grid(model, t_val)
        postProcess(0, 1.1, 0, 0.41,
                    [xs, ys, ts, u, v, p], s=2, num=i, outdir=outdir)

    print(f'\n  ✓  Run "{tag}" complete — figures in {outdir}/')
    return model


# ─────────────────────────────────────────────────────────────────────────────
# §9  Entry point — four runs
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    # Set retrain=True on any run to force retraining and overwrite its checkpoint.
    # Default (retrain=False): checkpoint is loaded and only figures are rebuilt.

    # Paper (transient): Ng=120k domain, Nd=9.6k Dirichlet, Nn=3.2k Neumann, Ni=3.5k IC
    # Adam: 10,000 iterations before L-BFGS-B (per paper)

    # (a) Default — paper-aligned
     run_experiment('run_a_default',
                   adam_iter=2000,
                   n_base=90000, n_refine=30000, n_lw=3000, n_up=3000)

    # (b) Fewer collocation points only (99% reduction of LHS base counts)
    # run_experiment('run_b_fewer_collo',
    #               adam_iter=10000,
    #               n_base=900, n_refine=300, n_lw=30, n_up=30)

    # (c) Fewer Adam iterations only (99% reduction: 10000→100)
    # run_experiment('run_c_fewer_adam',
    #               adam_iter=100,
    #               n_base=90000, n_refine=30000, n_lw=3000, n_up=3000)

    # (d) Both reductions together
    # run_experiment('run_d_both',
    #               adam_iter=100,
    #               n_base=900, n_refine=300, n_lw=30, n_up=30)


  RUN: run_a_default
  Collocation points: 140,799
  [fig] inlet_profile_3d.png
  [fig] collocation_distribution.png
  [runtime] Using TF2 GPU path
  Adam it     0  loss=8.168e-01  f=2.965e-01  W=6.076e-02  I=1.178e-01  O=1.251e-02  IC=6.906e-02
  Adam it   100  loss=1.790e-01  f=1.423e-02  W=2.271e-02  I=5.600e-02  O=4.395e-05  IC=3.622e-03
  Adam it   200  loss=1.550e-01  f=1.089e-02  W=2.751e-02  I=4.166e-02  O=3.335e-05  IC=2.868e-03
  Adam it   300  loss=1.409e-01  f=8.312e-03  W=2.846e-02  I=3.574e-02  O=2.145e-05  IC=2.074e-03
  Adam it   400  loss=1.319e-01  f=7.787e-03  W=2.338e-02  I=3.691e-02  O=1.651e-05  IC=1.762e-03
  Adam it   500  loss=1.208e-01  f=6.854e-03  W=2.695e-02  I=2.932e-02  O=8.662e-06  IC=6.763e-04
  Adam it   600  loss=1.161e-01  f=6.134e-03  W=2.671e-02  I=2.777e-02  O=9.350e-06  IC=4.991e-04
  Adam it   700  loss=1.120e-01  f=5.604e-03  W=2.529e-02  I=2.751e-02  O=6.766e-06  IC=3.965e-04
  Adam it   800  loss=1.098e-01  f=5.428e-03  W=2.687e-02  I=2.490e